# Introdução

### Etapa 1 — Importar bibliotecas

Antes de começarmos, precisamos importar duas bibliotecas fundamentais:

- **requests** → permite enviar requisições HTTP (para obter o conteúdo de páginas da web);
- **BeautifulSoup** (da biblioteca `bs4`) → usada para analisar (parsear) o HTML retornado e facilitar a extração de informações.
- **re** → usado para trabalhar com expressões regulares, que são sequências de caracteres que definem um padrão de busca em strings.

Execute o código abaixo:

In [54]:
import requests
from bs4 import BeautifulSoup
import re

### Etapa 2 — Definir o site que será utilizado

Vamos começar com um site criado especificamente para praticar *web scraping*:  
[https://quotes.toscrape.com](https://quotes.toscrape.com)

Sinta-se livre para escolher outro site.

In [2]:
url = "https://quotes.toscrape.com/"

### Etapa 3 — Fazer a requisição e verificar o status

Agora vamos usar `requests.get(url)` para obter o conteúdo da página.

O resultado dessa requisição é um **objeto `Response`**, que contém:
- O **código de status (status_code)** da resposta;
- O **conteúdo (response.text)** com o HTML bruto.

O `status_code` indica se a requisição foi bem-sucedida ou se houve algum erro.  
Veja os principais códigos e o que fazer em cada caso:

| Código | Significado | O que fazer |
|:-------:|:------------|:------------|
| **200** | **Sucesso** — o servidor respondeu corretamente e o conteúdo foi retornado. | Siga normalmente com a análise do HTML. |
| **301 / 302** | **Redirecionamento** — o site mudou de endereço ou está redirecionando. | Verifique `response.url` para ver a URL final. Caso precise seguir o redirecionamento manualmente, use `allow_redirects=True` (padrão no `requests`). |
| **400** | **Requisição inválida (Bad Request)** — o servidor não entendeu o pedido. | Confira se a URL está correta e se há parâmetros inválidos. |
| **401** | **Não autorizado (Unauthorized)** — o site exige autenticação. | Verifique se é necessário login ou token de acesso. |
| **403** | **Acesso proibido (Forbidden)** — o servidor bloqueou o acesso. | Tente alterar o *User-Agent* nos *headers* para simular um navegador real ou verifique o `robots.txt` do site. |
| **404** | **Não encontrado (Not Found)** — a página solicitada não existe. | Verifique se a URL está digitada corretamente. |
| **429** | **Muitas requisições (Too Many Requests)** — o site está limitando o número de acessos. | Espere alguns segundos (`time.sleep()`), reduza a frequência das requisições ou implemente um *delay* entre as chamadas. |
| **500** | **Erro interno do servidor (Internal Server Error)** — problema no servidor. | Tente novamente mais tarde. O erro está do lado do site. |
| **503** | **Serviço indisponível (Service Unavailable)** — o servidor está temporariamente fora do ar. | Espere alguns minutos e tente novamente. Pode indicar sobrecarga no servidor. |


In [55]:
headers = {
    "User-Agent": (
        'Mozilla/5.0 (Linux; Android 6.0.1; Nexus 5X Build/MMB29P) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.7034.0 Mobile Safari/537.36 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)'
    )
}

response = requests.get(url, headers=headers)

print("Status da resposta:", response.status_code)

# Verifica se foi bem-sucedido
if response.status_code == 200:
    print("Requisição bem-sucedida!")
else:
    print("Algo deu errado com a requisição.")

Status da resposta: 200
Requisição bem-sucedida!


### Etapa 4 — Criar o objeto BeautifulSoup

Com o HTML em mãos (`response.text`), o próximo passo é transformar esse texto em uma estrutura *navegável*.

O BeautifulSoup cria uma **árvore de objetos** que representa o HTML, permitindo acessar tags, atributos e conteúdos facilmente.

Escolhemos o *parser* `'html.parser'`, que é o nativo do Python e suficiente para a maioria dos casos.


In [4]:
soup = BeautifulSoup(response.text, "html.parser")

### Etapa 5 — Visualizar o HTML com `prettify()`

A função `prettify()` do BeautifulSoup formata o HTML com identação e quebras de linha, tornando a leitura muito mais fácil.

Como o HTML pode ser extenso, vamos exibir apenas os primeiros 1000 caracteres.


In [5]:
print(soup.prettify()[:1000])

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Quotes to Scrape
  </title>
  <link href="/static/bootstrap.min.css" rel="stylesheet"/>
  <link href="/static/main.css" rel="stylesheet"/>
 </head>
 <body>
  <div class="container">
   <div class="row header-box">
    <div class="col-md-8">
     <h1>
      <a href="/" style="text-decoration: none">
       Quotes to Scrape
      </a>
     </h1>
    </div>
    <div class="col-md-4">
     <p>
      <a href="/login">
       Login
      </a>
     </p>
    </div>
   </div>
   <div class="row">
    <div class="col-md-8">
     <div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
      <span class="text" itemprop="text">
       “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
      </span>
      <span>
       by
       <small class="author" itemprop="author">
        Albert Einstein
       </small>
       <a href="/author/Albert

### Etapa 6 — Acessando e extraindo tags específicas

Agora que já temos o HTML carregado e organizado com o BeautifulSoup, podemos começar a **navegar** pela estrutura e **extrair informações específicas**.

Cada **tag HTML** (como `<title>`, `<h1>`, `<p>`, `<a>`, etc.) é representada como um **objeto Python** dentro do `soup`.


In [6]:
# Vamos acessar a tag <title> da página
soup.title

<title>Quotes to Scrape</title>

In [8]:
tag = soup.title
type(tag)

bs4.element.Tag

Ao executar `soup.title`, o BeautifulSoup retorna o **objeto da tag `<title>`**.  
Podemos explorar propriedades muito úteis da **tag** para entender a estrutura do HTML.

In [9]:
print("Nome da tag:", soup.title.name)
print("Conteúdo da tag:", soup.title.string)

Nome da tag: title
Conteúdo da tag: Quotes to Scrape


Podemos acessar diretamente outras tags comuns, como `<h1>`, `<p>`, `<a>`, etc.

Por exemplo:
- `soup.h1` retorna o primeiro `<h1>` da página;
- `soup.p` retorna o primeiro parágrafo `<p>`;
- `soup.a` retorna o primeiro link `<a>` encontrado.

In [10]:
print("Título principal (h1):", soup.h1)
print("Primeiro parágrafo (p):", soup.p)
print("Primeiro link (a):", soup.a)

Título principal (h1): <h1>
<a href="/" style="text-decoration: none">Quotes to Scrape</a>
</h1>
Primeiro parágrafo (p): <p>
<a href="/login">Login</a>
</p>
Primeiro link (a): <a href="/" style="text-decoration: none">Quotes to Scrape</a>


### Entendendo o conteúdo de uma tag

Cada tag pode ter:
- **Conteúdo de texto** (acessado com `.text` ou `.string`);
- **Atributos** (como `href`, `class`, `id`, etc.), acessados como dicionário: `tag['atributo']`.

In [11]:
link = soup.a
print("Tag completa:", link)
print("Texto do link:", link.text)
print("Atributos da tag:", link.attrs)
print("Endereço (href):", link['href'])

Tag completa: <a href="/" style="text-decoration: none">Quotes to Scrape</a>
Texto do link: Quotes to Scrape
Atributos da tag: {'href': '/', 'style': 'text-decoration: none'}
Endereço (href): /


# Metodos do BeautifulSoup

### Etapa 7 — Busca de elementos com `find()` e `find_all()`

Após aprender a acessar tags específicas com `soup.tag`, a forma mais poderosa e flexível de localizar elementos é usar os métodos:

- `soup.find(name, attrs, recursive, text, **kwargs)` → retorna a **primeira** tag que satisfaça os critérios (ou `None`);
- `soup.find_all(name, attrs, recursive, text, limit, **kwargs)` → retorna **todas** as tags que satisfaçam os critérios (como uma lista).

Esses métodos permitem buscar por:
- **Nome da tag** (ex.: `'a'`, `'div'`, `'span'`);
- **Atributos** (ex.: `class_='intro'`, `id='main'`);
- **Texto interno** (ex.: `text='Next'` ou usando expressões regulares);
- **Filtragem mais complexa** (funções/predicados, `limit`, `recursive`, etc.).


In [12]:
first_a = soup.find('a')
all_a = soup.find_all('a')

print("Primeiro <a>:", first_a)
print("Quantidade de <a> encontradas:", len(all_a))

Primeiro <a>: <a href="/" style="text-decoration: none">Quotes to Scrape</a>
Quantidade de <a> encontradas: 55


#### Parâmetros importantes

- `name` → nome da tag (pode ser string, lista de strings, regex ou `True` para qualquer tag);
- `attrs` → dicionário com atributos ex.: `{'class': 'quote'}`;
- `class_` → para buscar pela classe (usar o underscore por causa da palavra reservada `class` em Python): `soup.find_all('div', class_='container')`;
- `id` → buscar por id: `soup.find(id='main')`;
- `text` → buscar texto exato ou regex: `soup.find_all('span', text='some text')` ou `re.compile(...)`;
- `limit` (em `find_all`) → número máximo de resultados;
- `recursive` (default `True`) → se `False` busca apenas filhos diretos;
- Você também pode passar uma função em `name` ou em `attrs` para filtragem customizada.


In [13]:
quotes = soup.find_all('span', class_='text')  # aqui usamos class_
authors = soup.find_all('small', class_='author')

print("Primeiras 3 citações:")
for q in quotes[:3]:
    print("-", q.get_text())

print("\nPrimeiros 3 autores:")
for a in authors[:3]:
    print("-", a.get_text())

Primeiras 3 citações:
- “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
- “It is our choices, Harry, that show what we truly are, far more than our abilities.”
- “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”

Primeiros 3 autores:
- Albert Einstein
- J.K. Rowling
- Albert Einstein


#### Buscar por atributos (href, title, data-*, etc.)

Para obter um atributo de cada tag:
- itere sobre `find_all(...)` e use `tag.get('atributo')` ou `tag['atributo']`.

Exemplo comum: extrair todos os `href` de links.


In [ ]:
# links = []
# for a in soup.find_all('a'):
#     links.append(a['href'])

links = [a.get('href') for a in soup.find_all('a')] # lista comprimida
links[:10]

['/',
 '/login',
 '/author/Albert-Einstein',
 '/tag/change/page/1/',
 '/tag/deep-thoughts/page/1/',
 '/tag/thinking/page/1/',
 '/tag/world/page/1/',
 '/author/J-K-Rowling',
 '/tag/abilities/page/1/',
 '/tag/choices/page/1/']

#### Exemplos avançados

1. **Buscar pela combinação de nome + classe + id**:

In [22]:
soup.find_all('div', class_='quote')

[<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
 <span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
 <span>by <small class="author" itemprop="author">Albert Einstein</small>
 <a href="/author/Albert-Einstein">(about)</a>
 </span>
 <div class="tags">
             Tags:
             <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
 <a class="tag" href="/tag/change/page/1/">change</a>
 <a class="tag" href="/tag/deep-thoughts/page/1/">deep-thoughts</a>
 <a class="tag" href="/tag/thinking/page/1/">thinking</a>
 <a class="tag" href="/tag/world/page/1/">world</a>
 </div>
 </div>,
 <div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
 <span class="text" itemprop="text">“It is our choices, Harry, that show what we truly are, far more than our abilities.”</span>
 <span>by <small class="author" itempr

2. **Buscar por várias tags (passar lista como name)**:

In [17]:
soup.find_all(['h1', 'h2', 'h3'])

[<h1>
 <a href="/" style="text-decoration: none">Quotes to Scrape</a>
 </h1>,
 <h2>Top Ten tags</h2>]

3. **Buscar texto com regex**:

In [19]:
soup.find_all('span', string=re.compile(r'^“'))  # citações que começam com aspas

[<span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>,
 <span class="text" itemprop="text">“It is our choices, Harry, that show what we truly are, far more than our abilities.”</span>,
 <span class="text" itemprop="text">“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”</span>,
 <span class="text" itemprop="text">“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”</span>,
 <span class="text" itemprop="text">“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”</span>,
 <span class="text" itemprop="text">“Try not to become a man of success. Rather become a man of value.”</span>,
 <span class="text" itemprop="text">“It is better to be hated for what you are than to be loved for what you are not.

4. **Usar função para filtragem**:

In [ ]:
def has_data_attr(tag):
    return tag.has_attr('data-preco')

soup.find_all(has_data_attr)

[]

5. **Limitar resultados**:

In [21]:
soup.find_all('a', limit=5)

[<a href="/" style="text-decoration: none">Quotes to Scrape</a>,
 <a href="/login">Login</a>,
 <a href="/author/Albert-Einstein">(about)</a>,
 <a class="tag" href="/tag/change/page/1/">change</a>,
 <a class="tag" href="/tag/deep-thoughts/page/1/">deep-thoughts</a>]

#### Outros Exemplos

In [26]:
authors_ab = soup.find_all('small', class_='author', string=re.compile(r'^[AB]')) # autores que o nome começa com 'A' ou 'B')
[a.get_text() for a in authors_ab]

['Albert Einstein', 'Albert Einstein', 'Albert Einstein', 'André Gide']

In [27]:
first_quote = soup.find('div', class_='quote')
quote_text = first_quote.find('span', class_='text').get_text()
quote_author = first_quote.find('small', class_='author').get_text()

print("Citação:", quote_text)
print("Autor:", quote_author)

Citação: “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
Autor: Albert Einstein


#### Boas práticas ao usar `find_all`

- Use `class_` em vez de `'class'` (evita conflito com palavra reservada).
- Prefira `get()` ao acessar atributos: `tag.get('href')` evita `KeyError` se não existir.
- Evite buscas desnecessariamente amplas (ex.: `soup.find_all(True)`) — filtre por tag ou atributo.
- Se o site tem muitas tags iguais, use `limit` ou processe o `find_all` em streaming (iteração) para economizar memória.


### Etapa 8 – Seletores CSS com `select()` e `select_one()`

Além dos métodos `find()` e `find_all()`, o **BeautifulSoup** também permite buscar elementos usando **seletores CSS**, que são amplamente utilizados em HTML e front-end.

Essa abordagem é poderosa e muitas vezes mais flexível, especialmente quando você quer buscar **por classes, IDs ou hierarquias de elementos**.


#### 1️ Usando `select_one()` — retorna **um único elemento**

O método `select_one()` busca **o primeiro elemento** que corresponde ao seletor CSS especificado.

In [28]:
titulo = soup.select_one('h1')
print(titulo.get_text(strip=True))

Quotes to Scrape


#### 2 Usando `select()` — retorna **uma lista de elementos**

O método `select` retorna **todos os elementos** que correspondem ao seletor.

In [29]:
links = soup.select('a')
for link in links:
    print(link.get_text(strip=True), "→", link.get('href'))

Quotes to Scrape → /
Login → /login
(about) → /author/Albert-Einstein
change → /tag/change/page/1/
deep-thoughts → /tag/deep-thoughts/page/1/
thinking → /tag/thinking/page/1/
world → /tag/world/page/1/
(about) → /author/J-K-Rowling
abilities → /tag/abilities/page/1/
choices → /tag/choices/page/1/
(about) → /author/Albert-Einstein
inspirational → /tag/inspirational/page/1/
life → /tag/life/page/1/
live → /tag/live/page/1/
miracle → /tag/miracle/page/1/
miracles → /tag/miracles/page/1/
(about) → /author/Jane-Austen
aliteracy → /tag/aliteracy/page/1/
books → /tag/books/page/1/
classic → /tag/classic/page/1/
humor → /tag/humor/page/1/
(about) → /author/Marilyn-Monroe
be-yourself → /tag/be-yourself/page/1/
inspirational → /tag/inspirational/page/1/
(about) → /author/Albert-Einstein
adulthood → /tag/adulthood/page/1/
success → /tag/success/page/1/
value → /tag/value/page/1/
(about) → /author/Andre-Gide
life → /tag/life/page/1/
love → /tag/love/page/1/
(about) → /author/Thomas-A-Edison
edison

#### 3 **Selecionando por classe**

Em CSS, as classes são referenciadas com um ponto `(.)`

In [31]:
itens = soup.select('.quote')
for i in itens:
    print(i.get_text(strip=True))

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”byAlbert Einstein(about)Tags:changedeep-thoughtsthinkingworld
“It is our choices, Harry, that show what we truly are, far more than our abilities.”byJ.K. Rowling(about)Tags:abilitieschoices
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”byAlbert Einstein(about)Tags:inspirationallifelivemiraclemiracles
“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”byJane Austen(about)Tags:aliteracybooksclassichumor
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”byMarilyn Monroe(about)Tags:be-yourselfinspirational
“Try not to become a man of success. Rather become a man of value.”byAlbert Einstein(about)Tags:adulthoodsuccessvalue
“It is better to be hated for what you are than to be loved 

#### 4 **Selecionando por ID**

IDs são referenciados com o símbolo `#`.

In [ ]:
soup.select_one('#main-exemplo')

#### 5 **Selecionando tags dentro de outras tags**

Podemos usar **hierarquia de seletores**, como em CSS.

In [35]:
links_tags = soup.select('div.tags a')
for link in links_tags:
    print(link.get_text(strip=True))

change
deep-thoughts
thinking
world
abilities
choices
inspirational
life
live
miracle
miracles
aliteracy
books
classic
humor
be-yourself
inspirational
adulthood
success
value
life
love
edison
failure
inspirational
paraphrased
misattributed-eleanor-roosevelt
humor
obvious
simile


### Etapa 9 – Navegação na Árvore DOM com BeautifulSoup

Uma das grandes vantagens do **BeautifulSoup** é permitir **navegar pela árvore DOM** — ou seja, caminhar entre **pais, filhos e irmãos** das tags.

Isso é útil quando queremos extrair dados que estão **próximos no HTML**, mas não têm um seletor claro.

#### 1 Acessando o elemento pai – `.parent`

O atributo `.parent` retorna o **elemento imediatamente acima** da tag atual na hierarquia HTML.

Se quiser subir mais níveis, você pode usar `.parent.parent`

In [36]:
link = soup.find('a')
for p in link.parents:
    print(p.name)

h1
div
div
div
body
html
[document]


#### 2 Acessando os filhos  – `.contents` e `.children`

O atributo `.contents` retorna uma lista com todos os filhos diretos da tag.

O atributo `.children` é um iterador (útil em loops grandes).

In [37]:
div = soup.find('div')
print(div.contents)  # Mostra os filhos diretos

['\n', <div class="row header-box">
<div class="col-md-8">
<h1>
<a href="/" style="text-decoration: none">Quotes to Scrape</a>
</h1>
</div>
<div class="col-md-4">
<p>
<a href="/login">Login</a>
</p>
</div>
</div>, '\n', <div class="row">
<div class="col-md-8">
<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
<span>by <small class="author" itemprop="author">Albert Einstein</small>
<a href="/author/Albert-Einstein">(about)</a>
</span>
<div class="tags">
            Tags:
            <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
<a class="tag" href="/tag/change/page/1/">change</a>
<a class="tag" href="/tag/deep-thoughts/page/1/">deep-thoughts</a>
<a class="tag" href="/tag/thinking/page/1/">thinking</a>
<a class="tag" href="/tag/world/page/1/">world</a>
</div>
</di

In [38]:
for filho in div.children:
    print(filho.name)

None
div
None
div
None


#### 3 Acessando todos os descendentes – `.descendants`

Enquanto `.children` traz apenas os filhos diretos, `.descendants` percorre todos os níveis abaixo da tag.

In [40]:
for desc in div.descendants:
    print(desc.name)

None
div
None
div
None
h1
None
a
None
None
None
None
div
None
p
None
a
None
None
None
None
None
div
None
div
None
div
None
span
None
None
span
None
small
None
None
a
None
None
None
div
None
meta
None
a
None
None
a
None
None
a
None
None
a
None
None
None
None
div
None
span
None
None
span
None
small
None
None
a
None
None
None
div
None
meta
None
a
None
None
a
None
None
None
None
div
None
span
None
None
span
None
small
None
None
a
None
None
None
div
None
meta
None
a
None
None
a
None
None
a
None
None
a
None
None
a
None
None
None
None
div
None
span
None
None
span
None
small
None
None
a
None
None
None
div
None
meta
None
a
None
None
a
None
None
a
None
None
a
None
None
None
None
div
None
span
None
None
span
None
small
None
None
a
None
None
None
div
None
meta
None
a
None
None
a
None
None
None
None
div
None
span
None
None
span
None
small
None
None
a
None
None
None
div
None
meta
None
a
None
None
a
None
None
a
None
None
None
None
div
None
span
None
None
span
None
small
None
None
a
None
None
None
div

#### 4 Acessando irmãos – `.next_sibling` e `.previous_sibling`

Esses atributos permitem navegar entre **tags que estão no mesmo nível** da árvore DOM.

In [ ]:
paragrafo = soup.find('div', class_='quote')
print("Parágrafo atual:", paragrafo)
print("Próximo irmão:", paragrafo.next_sibling.next_sibling)
# Às vezes os irmãos são apenas espaços ou quebras de linha, então é comum precisar pular valores None ou strings vazias.

Parágrafo atual: <div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
<span>by <small class="author" itemprop="author">Albert Einstein</small>
<a href="/author/Albert-Einstein">(about)</a>
</span>
<div class="tags">
            Tags:
            <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
<a class="tag" href="/tag/change/page/1/">change</a>
<a class="tag" href="/tag/deep-thoughts/page/1/">deep-thoughts</a>
<a class="tag" href="/tag/thinking/page/1/">thinking</a>
<a class="tag" href="/tag/world/page/1/">world</a>
</div>
</div>
Próximo irmão: <div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“It is our choices, Harry, that show what we truly are, far more than our abilities.”</span>
<span>by <small class="

#### 5 Acessando todas as próximas/anteriores tags

Para navegar apenas entre tags válidas (ignorando espaços), use:

- `.find_next_sibling()`;
- `.find_previous_sibling()`.

In [ ]:
span = soup.find('span', class_='text')
print(span.find_next_sibling('span'))
span.find_next()

<span>by <small class="author" itemprop="author">Albert Einstein</small>
<a href="/author/Albert-Einstein">(about)</a>
</span>
